In [1]:
import os
import json
import glob
import pandas as pd
import numpy as np

# ==========================================
# 1. 配置你要扫描的基础路径
# ==========================================
# 这里指向你模型结果的根目录，脚本会自动往下找所有包含 seed_ 的文件夹
base_search_dir = "/NAS/shith/uplift/results/criteo/train_y/TARNET/"

# ==========================================
# 2. 递归扫描所有的 seed 文件夹并提取指标
# ==========================================
print(f"🔍 正在扫描目录: {base_search_dir}")
# 匹配规则：往下任意层级找 seed_* 文件夹里的 final_metrics.json
search_pattern = os.path.join(base_search_dir, "**", "seed_*", "final_metrics.json")
metrics_files = glob.glob(search_pattern, recursive=True)

if not metrics_files:
    print("⚠️ 没有找到任何 final_metrics.json，请检查路径或确认模型是否已跑完落盘！")
else:
    print(f"✅ 共找到 {len(metrics_files)} 个 seed 实验结果，正在解析...\n")

data_records = []

for file_path in metrics_files:
    try:
        # 解析路径，提取实验名和 seed 号
        # 例如: .../y_ours_v8s6_h32_a0.1_t20_wd1e5/y_ours_v8s6_h32_a0.1_t20_wd1e5_mean/seed_42/final_metrics.json
        path_parts = file_path.split('/')
        seed_str = path_parts[-2]  # "seed_42"
        seed_num = seed_str.replace("seed_", "")
        
        # 提取实验名 (取 seed_xxx 上一级的上一级，即 y_ours_v8s6_h32_a0.1_t20_wd1e5)
        exp_name = path_parts[-4] 
        
        with open(file_path, 'r') as f:
            metrics = json.load(f)
            
        # 组装一条记录
        record = {
            "Experiment": exp_name,
            "Seed": seed_num,
        }
        # 提取关键指标 (你可以根据需要添加/删减你想看的列)
        key_metrics = [
            "Test_Target_Y_AUUC", 
            "Test_Target_C_AUUC", 
            "Test_Target_Y_AUC",
            "Test_Target_Y_MAE"
        ]
        
        for km in key_metrics:
            if km in metrics:
                record[km] = metrics[km]
                
        data_records.append(record)
        
    except Exception as e:
        print(f"❌ 解析文件失败: {file_path} | 报错: {e}")

# ==========================================
# 3. 数据聚合与展示
# ==========================================
# 转换为 DataFrame
df_all = pd.DataFrame(data_records)

if not df_all.empty:
    print("📊【单 Seed 原始数据明细】")
    display(df_all.sort_values(by=["Experiment", "Seed"]))
    
    # ------------------------------------------
    # 4. 计算多 Seed 的均值和标准差 (Mean ± Std)
    # ------------------------------------------
    # 过滤出所有数值类型的列
    numeric_cols = df_all.select_dtypes(include=[np.number]).columns.tolist()
    if "Seed" in numeric_cols:
        numeric_cols.remove("Seed") # 不对 Seed 求均值
        
    # 按实验名分组，计算均值和标准差
    df_agg = df_all.groupby("Experiment")[numeric_cols].agg(['mean', 'std']).reset_index()
    
    # 格式化输出: 把 mean 和 std 拼成 "0.9123 ± 0.0012" 的美观格式
    df_formatted = pd.DataFrame()
    df_formatted["Experiment"] = df_agg["Experiment"]
    
    for col in numeric_cols:
        # 获取 mean 和 std 列，并保留 4 位小数
        means = df_agg[(col, 'mean')].round(4).astype(str)
        stds = df_agg[(col, 'std')].round(4).astype(str)
        df_formatted[col] = means + " ± " + stds

    print("\n" + "="*80)
    print("🏆【多 Seed 聚合结果 (Mean ± Std)】")
    print("="*80)
    display(df_formatted)

🔍 正在扫描目录: /NAS/shith/uplift/results/criteo/train_y/TARNET/
✅ 共找到 29 个 seed 实验结果，正在解析...

📊【单 Seed 原始数据明细】


,Experiment,Seed,Test_Target_Y_AUUC,Test_Target_C_AUUC,Test_Target_Y_AUC,Test_Target_Y_MAE
28,y_ours_v8s6_h16_a0.1_t1_wd1e5,42,0.900334,0.823023,0.899419,0.006251
23,y_ours_v8s6_h32_a0.1_t20_wd1e5,1042,0.896977,0.835925,0.889213,0.006339
22,y_ours_v8s6_h32_a0.1_t20_wd1e5,42,0.905761,0.818161,0.894650,0.006861
24,y_ours_v8s6_h32_a0.5_t20_wd1e5,1042,0.892369,0.832797,0.897706,0.006942
25,y_ours_v8s6_h32_a0.5_t20_wd1e5,42,0.904948,0.818997,0.896551,0.008100
6,y_ours_v8s6_h32_a10.0_t1_wd0.01,1042,0.810891,0.710317,0.202078,0.013458
7,y_ours_v8s6_h32_a10.0_t1_wd0.01,42,0.904789,0.820537,0.890333,0.032406
12,y_ours_v8s6_h32_a10.0_t20_wd0.01,1042,0.810891,0.710317,0.202078,0.013458
13,y_ours_v8s6_h32_a10.0_t20_wd0.01,42,0.904789,0.820537,0.890333,0.032406
20,y_ours_v8s6_hNone_a1.0_t1_wd0.001,1042,0.892205,0.833842,0.818568,0.008357



🏆【多 Seed 聚合结果 (Mean ± Std)】


,Experiment,Test_Target_Y_AUUC,Test_Target_C_AUUC,Test_Target_Y_AUC,Test_Target_Y_MAE
0,y_ours_v8s6_h16_a0.1_t1_wd1e5,0.9003 ± nan,0.823 ± nan,0.8994 ± nan,0.0063 ± nan
1,y_ours_v8s6_h32_a0.1_t20_wd1e5,0.9014 ± 0.0062,0.827 ± 0.0126,0.8919 ± 0.0038,0.0066 ± 0.0004
2,y_ours_v8s6_h32_a0.5_t20_wd1e5,0.8987 ± 0.0089,0.8259 ± 0.0098,0.8971 ± 0.0008,0.0075 ± 0.0008
3,y_ours_v8s6_h32_a10.0_t1_wd0.01,0.8578 ± 0.0664,0.7654 ± 0.0779,0.5462 ± 0.4867,0.0229 ± 0.0134
4,y_ours_v8s6_h32_a10.0_t20_wd0.01,0.8578 ± 0.0664,0.7654 ± 0.0779,0.5462 ± 0.4867,0.0229 ± 0.0134
5,y_ours_v8s6_hNone_a1.0_t1_wd0.001,0.8964 ± 0.0059,0.8321 ± 0.0024,0.8423 ± 0.0336,0.009 ± 0.0009
6,y_ours_v8s6_hNone_a1.0_t20_wd1e5,0.8916 ± 0.0051,0.8272 ± 0.0006,0.8832 ± 0.0191,0.0078 ± 0.0007
7,y_pure_v10_h16_a1.0_wd1e4,0.8789 ± 0.0161,0.78 ± 0.0588,0.5433 ± 0.5736,0.0503 ± 0.0628
8,y_pure_v10_h16_a5.0_wd1e5,0.8811 ± 0.02,0.7858 ± 0.0665,0.5412 ± 0.5735,0.0515 ± 0.0607
9,y_pure_v10_h32_a1.0_wd1e4,0.8997 ± 0.0015,0.8275 ± 0.0084,0.9104 ± 0.0112,0.0073 ± 0.0007
